# Part 1 — Deterministic Multi-Period Supply Chain MILP

### Formulation choices, made visible as numbers

A deliberately small 3-tier, 2-region network used to isolate four modelling decisions that
are usually argued about rather than measured:

| § | Question | What it isolates |
|---|---|---|
| 3 | How much does each year matter? | Discount rate sets the **effective** model length |
| 4 | Lump-sum vs annualized capex | End-of-horizon truncation bias |
| 5 | Investment period granularity | Binaries vs quality; **direction of the bound** |
| 6 | No / exogenous / endogenous learning | The "free lunch" bias; SOS2 linearization |
| 7 | Perfect foresight vs rolling horizon | When $W<T$ matters; the tail-ban artifact |

**Part 2** (separate notebook) replaces the deterministic demand with a scenario tree and
compares the Expected-Value, Perfect-Information and Stochastic strategies, then solves the
stochastic program with Progressive Hedging.


**Network.** Mining → Processing → Manufacturing, two sites per tier, serving two demand
regions. Intra-region transport is cheap, cross-region is 4× dearer. Region 2 grows much
faster than Region 1, so the optimal answer shifts the network's centre of gravity over
time. Legacy assets retire on a **staggered** schedule (years 7/10/12/15/16/19), forcing
replacement waves throughout the horizon rather than one cliff at the end.

Facility siting is the binary part: `y[s,v,k]` = build unit *k* at site *s* decided in
year *v*, with a per-site construction lead time.

**Runtime:** ~5–8 minutes. §4's annual case is the slow one (~40 s) — that slowness is the
point being demonstrated.

## Legend / Notation

### Cost units in this instance
All costs are arbitrary money units, calibrated so the tradeoffs bite:

| Item | Magnitude |
|---|---|
| Build one facility unit (`capex0`) | 1,800 – 5,000 |
| Capacity per unit built (`cap_unit`) | 90 – 110 throughput units/yr |
| Variable cost, one delivered unit (all 3 tiers, ~53% chain yield) | ~10 – 13 |
| Transport per unit | 0.4 intra-region, 1.6 cross-region |
| Unmet-demand penalty (`slack_pen`) | 45 default; swept 30 – 250 in Part 2 |

### Time and horizon
| Symbol | Meaning |
|---|---|
| `T` | horizon length (years) |
| `t` | operating year |
| `v` | **vintage** — the year an asset's build was decided |
| `r` | discount rate (5% here) |
| $\delta_t = 1/(1+r)^t$ | discount factor |
| `invest_years` / `iy` | years in which a build decision may be taken |
| `L` (`d.life`) | asset life (20 yr) |
| `lead` | construction lead time (2–3 yr by site) |

### Capital cost
| Symbol | Meaning |
|---|---|
| `CRF` | capital recovery factor, $\dfrac{r(1+r)^L}{(1+r)^L-1}$ — converts a lump sum into an equivalent annuity |
| `lumpsum` | charge the full capital cost at the decision year |
| `annualized` | charge `CRF` × cost in each operating year inside the horizon |

### Efficiency
| Symbol | Meaning |
|---|---|
| $\eta(v,t)$ | yield of a vintage-$v$ asset operating in year $t$ |
| $\bar\eta$ | ceiling (thermodynamic / technical limit) |
| $\alpha$ | frontier improvement rate — applies to **new builds** |
| $\beta$ | within-life improvement rate for an **existing** asset, $\beta < \alpha$ |
| $\bar\Delta$ | cap on total lifetime retrofit gain for one asset |

### Learning
| Symbol | Meaning |
|---|---|
| `LR` | learning rate — fractional cost drop per **doubling** of cumulative capacity |
| $b = -\log_2(1-LR)$ | Wright's law exponent |
| $Q$, $Q_0$ | cumulative capacity; incumbent base |
| $C(Q)$ | **cumulative** capex — this is what gets linearized, not unit cost |
| $\lambda$ | SOS2 interpolation weights on the piecewise curve |

### Rolling horizon
| Symbol | Meaning |
|---|---|
| `W` | foresight window — how far ahead each solve can see |
| `delta` ($\Delta$) | roll step — years **committed** before re-solving |
| `decision_zone` (`dz`) | years inside the window where binary builds are allowed |
| `tail_continuous` | beyond `dz`, capacity is continuous (recommended) rather than banned |

## 1. Setup

The `pip` build of Gurobi ships a **restricted licence capped near 2,000 variables and
2,000 constraints**. Every model here stays under it. If you scale `T` or add sites you
will hit the cap — swap in your university's WLS licence:

```python
env = gp.Env(params={"WLSACCESSID": "...", "WLSSECRET": "...", "LICENSEID": 123456})
```
and pass `env=env` into `gp.Model()` inside `build()`.

In [ ]:
!pip install gurobipy --quiet
import math, time, random, itertools
import gurobipy as gp
from gurobipy import GRB
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
print("gurobipy", gp.gurobi.version())

## 2. Data and model

### Efficiency: vintage frontier + within-life improvement

Both effects are **parameters**, not variables — precomputed and indexed into.

$$\eta_{\text{new}}(v) = \bar\eta - (\bar\eta - \eta_0)(1-\alpha)^{v-v_0}$$
$$\eta(v,t) = \min\Big\{\eta_{\text{new}}(v) + \bar\Delta,\;\; \bar\eta - (\bar\eta - \eta_{\text{new}}(v))(1-\beta)^{t-v}\Big\}, \qquad \beta < \alpha$$

An older asset improves over its life but **never overtakes a newer vintage** — it starts
further back and closes the same gap more slowly — and nothing crosses the ceiling
$\bar\eta$. The `min` caps total retrofit gain at $\bar\Delta$.

Efficiency enters the **constraint matrix** (feedstock per unit of output), not the
objective. That is why no cost parameter can substitute for it: it changes the feasible
region and propagates upstream through every flow balance.

In [ ]:
class Data:
    def __init__(self, T=20, r=0.05):
        self.T = T
        self.r = r
        self.years = list(range(1, T + 1))
        self.df = {t: 1.0 / (1 + r) ** t for t in self.years}

        self.regions = ['R1', 'R2']
        self.mines = ['M1', 'M2']
        self.procs = ['P1', 'P2']
        self.fabs  = ['F1', 'F2']
        self.sites = self.mines + self.procs + self.fabs
        # home region of each site
        self.home = {'M1':'R1','M2':'R2','P1':'R1','P2':'R2','F1':'R1','F2':'R2'}

        self.cap_unit = {'M1':110,'M2':110,'P1':100,'P2':100,'F1':90,'F2':90}
        self.max_builds = 3
        self.life = 20            # asset life (yrs)
        self.lead = {'M1':2,'M2':2,'P1':3,'P2':3,'F1':2,'F2':2}

        # capex per build (year-0 cost, before learning)
        self.capex0 = {'M1':1800,'M2':2000,'P1':3300,'P2':3100,'F1':2900,'F2':2700}
        # variable opex per unit throughput
        self.opex = {'M1':1.2,'M2':1.4,'P1':2.0,'P2':2.2,'F1':2.5,'F2':2.3}

        # transport cost per unit: cheap intra-region, dear cross-region
        self.tc = lambda a, b: 0.4 if self.home[a] == self.home[b] else 1.6
        self.tc_dem = lambda f, rg: 0.4 if self.home[f] == rg else 1.6

        # ---- efficiency (yield) : vintage frontier + within-life improvement ----
        self.eta_bar = {'P':0.95, 'F':0.93}   # ceiling
        self.eta_0   = {'P':0.80, 'F':0.78}   # vintage-1 frontier
        self.alpha   = {'P':0.030,'F':0.025}  # frontier improvement / yr
        self.beta    = {'P':0.010,'F':0.008}  # within-life improvement / yr
        self.dbar    = {'P':0.05, 'F':0.05}   # max lifetime gain for one asset

        # mining yield is constant (ore -> concentrate)
        self.eta_mine = 0.90
        self.eta_min = 0.60   # clamp: legacy assets can't be arbitrarily bad

        # brownfield: legacy units already operating in year 1 (vintage <= 0)
        # legacy: (units, vintage, retirement year). Staggered retirements create
        # replacement waves spread across the horizon, not one cliff at the end.
        self.legacy = {'M1':(2,-6, 7), 'M2':(2,-6, 10),
                       'P1':(2,-3,12), 'P2':(2,-3, 15),
                       'F1':(2,-1,16), 'F2':(2,-1, 19)}

        # ---- demand: R2 grows much faster than R1 ----
        self.D = {}
        for t in self.years:
            self.D['R1', t] = 110 * (1.025 ** (t - 1))
            self.D['R2', t] = 85 * (1.055 ** (t - 1))
        self.slack_pen = 45.0     # unmet demand penalty per unit

        # ---- learning ----
        self.learn_frac = 0.70    # share of capex that is learnable technology
        self.learn_sites = self.procs + self.fabs   # recovery tech learns; mining is mature
        self.LR = 0.20            # 15% cost drop per doubling of cumulative capacity
        self.Q0 = 380.0           # incumbent cumulative capacity
        self.c_floor_frac = 0.55  # floor as fraction of c0
        self.g_exog = 0.035       # exogenous capex decline / yr (for 'exogenous' mode)

    def eta(self, tier, v, t):
        """Yield of a tier-`tier` asset of vintage v, operating in year t."""
        eb, e0 = self.eta_bar[tier], self.eta_0[tier]
        a, b, db = self.alpha[tier], self.beta[tier], self.dbar[tier]
        e_new = eb - (eb - e0) * (1 - a) ** (v - 1)
        e_new = max(e_new, self.eta_min)
        e_t   = eb - (eb - e_new) * (1 - b) ** (t - v)
        return max(self.eta_min, min(e_new + db, e_t))

    def crf(self):
        r, L = self.r, self.life
        return r * (1 + r) ** L / ((1 + r) ** L - 1)

    def tier_of(self, s):
        return 'M' if s in self.mines else ('P' if s in self.procs else 'F')

### Capex PV multipliers

- **`lumpsum`** — the whole \$1 paid at the decision year: multiplier $=\delta_v$.
- **`annualized`** — $\text{CRF}\times\$1$ charged each operating year inside the horizon,
  $\text{CRF} = \dfrac{r(1+r)^L}{(1+r)^L-1}$.

Because the annuity discounts back to exactly \$1 over a full life, the fraction of **cost**
charged inside the horizon equals the fraction of **value** captured — the truncation bias
cancels identically for a flat benefit stream. §3 measures what happens when it doesn't.

In [ ]:
# ---------------- CAPEX PV MULTIPLIERS ----------------
def capex_pv_multiplier(d, s, dec_year, mode, y_start=1, y_end=None):
    """PV (discounted to yr 0) of $1 of capex for a facility decided in `dec_year`.

    annualized : CRF x $1 charged each operating yr inside [y_start, y_end]
    lumpsum    : full $1 paid at the decision year
    """
    y_end = y_end or d.T
    online = dec_year + d.lead[s]
    if mode == 'lumpsum':
        return d.df.get(dec_year, 0.0)
    last = min(online + d.life - 1, y_end)
    if last < online:
        return 0.0
    return d.crf() * sum(d.df[t] for t in range(online, last + 1) if t in d.df)

### Learning curve — and why SOS2 is mandatory

Wright's law on cumulative capacity, with a floor:
$$c(Q) = \max\{c_{\text{floor}},\; c_0 (Q/Q_0)^{-b}\}, \qquad b = -\log_2(1-LR)$$

The trap is linearizing the **unit** cost. You must linearize the **cumulative** cost
$C(Q)=\int_{Q_0}^{Q}c(q)\,dq$ and take differences between periods.

$C(Q)$ is **concave increasing** and we **minimize**, so an LP relaxation with free
$\lambda$ rides the global chord — which lies *below* the true curve and hands the model a
cost reduction it never earned. **SOS2 restricts $\lambda$ to at most two adjacent
nonzeros**, which is what makes the approximation valid. That restriction is also what
costs branch-and-bound nodes.

In [ ]:
# ---------------- LEARNING BREAKPOINTS ----------------
def learning_breakpoints(d, qmax, nbp=7):
    """Breakpoints for cumulative capex C(Q) under Wright's law with a floor."""
    b = -math.log2(1 - d.LR)
    c0 = 1.0                                   # normalised unit cost multiplier
    cf = d.c_floor_frac

    def unit(q):
        return max(cf, c0 * (q / d.Q0) ** (-b))

    def cum(q):
        # integral of unit() from Q0 to q, done numerically (robust w/ the floor)
        n, lo, acc = 400, d.Q0, 0.0
        if q <= lo:
            return 0.0
        h = (q - lo) / n
        for i in range(n):
            acc += 0.5 * (unit(lo + i * h) + unit(lo + (i + 1) * h)) * h
        return acc

    Qbp = [d.Q0 + (qmax - d.Q0) * (i / (nbp - 1)) for i in range(nbp)]
    Cbp = [cum(q) for q in Qbp]
    return Qbp, Cbp, unit

### The model

Capital cost splits into a **site adder** (permitting, site prep — never learns) and a
**technology component** (learnable). Learning applies to processing and manufacturing,
which share a modular recovery technology; mining is mature.

This split does two jobs: it keeps the three learning modes cost-comparable, and it keeps
endogenous learning **linear** — the SOS2 output multiplies a scalar PV factor, never a
variable.

Note `thr[s,v,t]`: throughput is indexed by **vintage**, not just operating year. That
indexing is what makes efficiency, retirement and replacement economics work. An $\eta(t)$
indexed only on operating year would silently upgrade existing assets and destroy the value
of replacement decisions.

In [ ]:
# ---------------- MODEL BUILDER ----------------
def build(d, invest_years=None, capex_mode='annualized', learning='none',
          y_start=1, y_end=None, fixed_builds=None, forced_zero_after=None,
          relax_int_after=None, quiet=True, demand=None, into=None, prefix=''):
    """
    invest_years      : years in which a build decision may be taken
    capex_mode        : 'annualized' | 'lumpsum'
    learning          : 'none' | 'exogenous' | 'endogenous'
    y_start,y_end     : operating window (for rolling horizon)
    fixed_builds      : {(site, dec_year): 0/1} decisions frozen from earlier rolls
    forced_zero_after : no new builds decided after this year
    relax_int_after   : builds decided after this year are continuous in [0,1]
    """
    y_end = y_end or d.T
    yrs = [t for t in d.years if y_start <= t <= y_end]
    if invest_years is None:
        invest_years = list(d.years)
    IY = [v for v in invest_years if y_start <= v <= y_end]
    if forced_zero_after is not None:
        IY = [v for v in IY if v <= forced_zero_after]

    def usable(s, v):
        # a decision is only meaningful if the asset comes online inside the window
        return v + d.lead[s] <= y_end
    site_IY = {s: [v for v in IY if usable(s, v)] for s in d.sites}

    D = demand if demand is not None else d.D
    if into is not None:
        m = into
    else:
        m = gp.Model()
        if quiet:
            m.Params.OutputFlag = 0
        m.Params.MIPGap = 0.005

    # --- build decisions: y[s,v,k] = k-th unit at site s decided in year v ---
    idx = [(s, v, k) for s in d.sites for v in site_IY[s] for k in range(d.max_builds)]
    y = {}
    for (s, v, k) in idx:
        cont = (relax_int_after is not None and v > relax_int_after)
        y[s, v, k] = m.addVar(vtype=GRB.CONTINUOUS if cont else GRB.BINARY,
                              ub=1.0, name=f"{prefix}y_{s}_{v}_{k}")
    # symmetry breaking within a site-year
    for s in d.sites:
        for v in site_IY[s]:
            for k in range(d.max_builds - 1):
                m.addConstr(y[s, v, k] >= y[s, v, k + 1])

    if fixed_builds:
        for (s, v), val in fixed_builds.items():
            for k in range(d.max_builds):
                if (s, v, k) in y:
                    pass
        # handled by caller via 'prebuilt' capacity instead

    # --- prebuilt capacity inherited from earlier rolls: {(site, online_yr, vintage): units}
    prebuilt = fixed_builds or {}

    def online_units(s, t):
        """Expression for number of units of site s available in year t, by vintage."""
        terms = []
        if s in d.legacy:
            ln, lv, lret = d.legacy[s]
            if t <= lret:
                terms.append((lv, float(ln)))
        for v in site_IY[s]:
            on = v + d.lead[s]
            if on <= t <= on + d.life - 1:
                terms.append((v, gp.quicksum(y[s, v, k] for k in range(d.max_builds))))
        for (ps, pv), n in prebuilt.items():
            if ps == s:
                on = pv + d.lead[s]
                if on <= t <= on + d.life - 1:
                    terms.append((pv, n))
        return terms

    # --- throughput, vintage indexed at P and F tiers ---
    thr = {}
    for s in d.procs + d.fabs:
        for t in yrs:
            for (v, _) in online_units(s, t):
                if (s, v, t) not in thr:
                    thr[s, v, t] = m.addVar(name=f"thr_{s}_{v}_{t}")
    ext = {(s, t): m.addVar(name=f"ext_{s}_{t}") for s in d.mines for t in yrs}

    # --- arc flows ---
    fmp = {(a, b, t): m.addVar() for a in d.mines for b in d.procs for t in yrs}
    fpf = {(a, b, t): m.addVar() for a in d.procs for b in d.fabs for t in yrs}
    ffr = {(a, g, t): m.addVar() for a in d.fabs for g in d.regions for t in yrs}
    slk = {(g, t): m.addVar() for g in d.regions for t in yrs}

    # --- capacity constraints ---
    for t in yrs:
        for s in d.mines:
            cap = gp.quicksum(n * d.cap_unit[s] for (_, n) in online_units(s, t))
            m.addConstr(ext[s, t] <= cap)
        for s in d.procs + d.fabs:
            for (v, n) in online_units(s, t):
                m.addConstr(thr[s, v, t] <= n * d.cap_unit[s])

    # --- flow balances ---
    for t in yrs:
        for s in d.mines:
            m.addConstr(d.eta_mine * ext[s, t] == gp.quicksum(fmp[s, b, t] for b in d.procs))
        for s in d.procs:
            vints = [v for (v, _) in online_units(s, t)]
            m.addConstr(gp.quicksum(fmp[a, s, t] for a in d.mines)
                        == gp.quicksum(thr[s, v, t] for v in vints))
            m.addConstr(gp.quicksum(d.eta('P', v, t) * thr[s, v, t] for v in vints)
                        == gp.quicksum(fpf[s, b, t] for b in d.fabs))
        for s in d.fabs:
            vints = [v for (v, _) in online_units(s, t)]
            m.addConstr(gp.quicksum(fpf[a, s, t] for a in d.procs)
                        == gp.quicksum(thr[s, v, t] for v in vints))
            m.addConstr(gp.quicksum(d.eta('F', v, t) * thr[s, v, t] for v in vints)
                        == gp.quicksum(ffr[s, g, t] for g in d.regions))
        for g in d.regions:
            m.addConstr(gp.quicksum(ffr[f, g, t] for f in d.fabs) + slk[g, t] >= D[g, t])

    # --- capex term ---
    # capex(s,v) = site adder (never learns) + technology cost (may learn)
    LS = set(d.learn_sites)
    adder = {s: d.capex0[s] * (1 - d.learn_frac) if s in LS else d.capex0[s]
             for s in d.sites}
    tech_rate = (sum(d.capex0[s] * d.learn_frac / d.cap_unit[s] for s in LS)
                 / len(LS))                       # $ per unit of capacity

    capex_expr = gp.LinExpr()
    for s in d.sites:                              # site adders, all modes
        for v in site_IY[s]:
            mult = capex_pv_multiplier(d, s, v, capex_mode, y_start, y_end)
            for k in range(d.max_builds):
                capex_expr += mult * adder[s] * y[s, v, k]

    if learning in ('none', 'exogenous'):
        for s in LS:
            for v in site_IY[s]:
                mult = capex_pv_multiplier(d, s, v, capex_mode, y_start, y_end)
                decay = (1 - d.g_exog) ** (v - 1) if learning == 'exogenous' else 1.0
                rate = tech_rate * max(decay, d.c_floor_frac)
                for k in range(d.max_builds):
                    capex_expr += mult * rate * d.cap_unit[s] * y[s, v, k]
    else:  # endogenous: SOS2 on CUMULATIVE technology cost
        qmax = d.Q0 + sum(d.cap_unit[s] * d.max_builds for s in LS) * max(
            1, len(IY) // 3)
        Qbp, Cbp, _ = learning_breakpoints(d, qmax)
        allv = sorted(set(v for s in LS for v in site_IY[s]))
        prevC = None
        for v in allv:
            Qv = m.addVar(lb=d.Q0, ub=qmax, name=f"Qcum_{v}")
            Cv = m.addVar(lb=0, name=f"Ccum_{v}")
            lam = [m.addVar(lb=0, ub=1, name=f"lam_{v}_{j}") for j in range(len(Qbp))]
            m.addConstr(gp.quicksum(lam) == 1)
            m.addConstr(Qv == gp.quicksum(l * q for l, q in zip(lam, Qbp)))
            m.addConstr(Cv == gp.quicksum(l * c for l, c in zip(lam, Cbp)))
            m.addSOS(GRB.SOS_TYPE2, lam)           # <-- the essential restriction
            pre_q = sum(d.cap_unit[ps] * n for (ps, pv), n in prebuilt.items()
                        if ps in LS and pv <= v)
            m.addConstr(Qv == d.Q0 + pre_q + gp.quicksum(
                d.cap_unit[s] * y[s, vv, k]
                for s in LS for vv in site_IY[s] if vv <= v
                for k in range(d.max_builds)))
            cand = [capex_pv_multiplier(d, s, v, capex_mode, y_start, y_end)
                    for s in LS if v in site_IY[s]]
            mult = sum(cand) / len(cand) if cand else 0.0
            capex_expr += mult * tech_rate * (Cv - (prevC if prevC is not None else 0))
            prevC = Cv

    # --- operating + transport + penalty ---
    op = gp.LinExpr()
    for t in yrs:
        w = d.df[t]
        for s in d.mines:
            op += w * d.opex[s] * ext[s, t]
        for s in d.procs + d.fabs:
            for (v, _) in online_units(s, t):
                op += w * d.opex[s] * thr[s, v, t]
        for a in d.mines:
            for b in d.procs:
                op += w * d.tc(a, b) * fmp[a, b, t]
        for a in d.procs:
            for b in d.fabs:
                op += w * d.tc(a, b) * fpf[a, b, t]
        for f in d.fabs:
            for g in d.regions:
                op += w * d.tc_dem(f, g) * ffr[f, g, t]
        for g in d.regions:
            op += w * d.slack_pen * slk[g, t]

    if into is None:
        m.setObjective(capex_expr + op, GRB.MINIMIZE)
    m._y, m._siteIY, m._IY, m._slk, m._ffr, m._d = y, site_IY, IY, slk, ffr, d
    m._capex_expr, m._op_expr = capex_expr, op
    m._adder, m._tech_rate = adder, tech_rate
    if into is not None:
        return m, y, capex_expr + op, slk
    return m


def build_plan(m):
    """Extract {(site, decision_year): n_units} from a solved model."""
    out = {}
    for (s, v, k), var in m._y.items():
        if var.X > 0.5:
            out[s, v] = out.get((s, v), 0) + 1
    return out

In [ ]:
import time


def solve(d, tag, **kw):
    t0 = time.time()
    m = build(d, **kw)
    m.optimize()
    el = time.time() - t0
    if m.Status not in (GRB.OPTIMAL, GRB.TIME_LIMIT) or m.SolCount == 0:
        return dict(tag=tag, obj=None, sec=el, status=m.Status)
    return dict(tag=tag, obj=m.ObjVal, bound=m.ObjBound, sec=el,
                nbin=m.NumBinVars, nvar=m.NumVars,
                nodes=int(m.NodeCount),
                plan=build_plan(m),
                slack=sum(v.X for v in m._slk.values()),
                model=m)


def evaluate_plan(d, plan, capex_mode='annualized', learning='none'):
    """Cost of a FIXED build plan under the full-horizon model (ops re-optimised)."""
    m = build(d, invest_years=[], capex_mode=capex_mode,
              learning=learning, fixed_builds=plan)
    # add back the capex of the fixed plan (build() charges nothing for prebuilt)
    extra = 0.0
    LS = set(d.learn_sites)
    tech_rate = (sum(d.capex0[x] * d.learn_frac / d.cap_unit[x] for x in LS) / len(LS))
    for (s, v), n in plan.items():
        mult = capex_pv_multiplier(d, s, v, capex_mode)
        unit = (d.capex0[s] * (1 - d.learn_frac) + tech_rate * d.cap_unit[s]
                if s in LS else d.capex0[s])
        extra += n * unit * mult
    m.optimize()
    if m.SolCount == 0:
        return None
    return m.ObjVal + extra


def rolling_horizon(d, W, delta, invest_step=1, decision_zone=None,
                    tail_continuous=True, capex_mode='annualized'):
    """
    W              : foresight window length
    delta          : roll step (years committed per solve)
    decision_zone  : years from window start in which BINARY builds are allowed
                     (None = whole window)
    tail_continuous: builds beyond the decision zone are continuous, not banned
    Returns the committed plan and diagnostics.
    """
    committed = {}
    log = []
    start = 1
    while start <= d.T:
        y_end = min(start + W - 1, d.T)
        dz_end = y_end if decision_zone is None else min(start + decision_zone - 1, y_end)
        iy = [v for v in range(start, y_end + 1) if (v - 1) % invest_step == 0]
        kw = dict(invest_years=iy, capex_mode=capex_mode,
                  y_start=start, y_end=y_end, fixed_builds=dict(committed))
        if tail_continuous:
            kw['relax_int_after'] = dz_end
        else:
            kw['forced_zero_after'] = dz_end
        m = build(d, **kw)
        m.optimize()
        if m.SolCount == 0:
            log.append((start, y_end, 'INFEASIBLE'))
            break
        # commit only decisions inside [start, start+delta-1]
        newly = {}
        for (s, v, k), var in m._y.items():
            if v <= start + delta - 1 and var.X > 0.5:
                newly[s, v] = newly.get((s, v), 0) + 1
        for key, n in newly.items():
            committed[key] = committed.get(key, 0) + n
        log.append((start, y_end, dz_end, dict(newly)))
        start += delta
    return committed, log


def staggered_years(T, fine=6, mid_step=2, mid_end=10, coarse_step=5):
    """Annual for yrs 1..fine, every mid_step to mid_end, then every coarse_step."""
    ys = list(range(1, fine + 1))
    ys += [t for t in range(fine + 1, mid_end + 1) if (t - fine - 1) % mid_step == 0]
    ys += [t for t in range(mid_end + 1, T + 1) if (t - mid_end - 1) % coarse_step == 0]
    return sorted(set(ys))


## 3. How much does each year actually matter?

Before any of the formulation choices, establish the ground the rest of the notebook stands
on. Take a flat \$1,000 cost every year forever and ask what each year contributes.

**The discount rate sets the *effective* length of your model** — independently of the `T`
you type in. This is why horizon truncation, late-build accounting, and rolling-horizon
window length are all the *same* question wearing different hats.


In [ ]:
C, T100, rates = 1000.0, 100, [0.03, 0.05, 0.10]
cum = {r: np.cumsum([C/(1+r)**t for t in range(1, T100+1)]) for r in rates}
rows = []
for y in list(range(1, 11)) + list(range(15, 101, 5)):
    row = {'year': y}
    for r in rates:
        row[f'PV@{int(r*100)}%']  = round(C/(1+r)**y, 1)
        row[f'cum@{int(r*100)}%'] = round(cum[r][y-1], 0)
    rows.append(row)
pd.DataFrame(rows)

In [ ]:
print(" r    100-yr NPV   perpetuity cap   90% of cap reached by   'effective model'")
for r in rates:
    hit = next(t for t in range(1, T100+1) if cum[r][t-1] >= 0.90*C/r)
    print(f"{int(r*100):2d}%   {cum[r][-1]:10,.0f}   {C/r:14,.0f}   {hit:21d}   ~{hit}-year model")
print("\\nUndiscounted 100-yr total would be 100,000.")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.6))
cols = {0.03:'#2471a3', 0.05:'#196f3d', 0.10:'#c0392b'}
for r in rates:
    ax[0].plot(range(1,T100+1),[C/(1+r)**t for t in range(1,T100+1)],lw=2.8,
               color=cols[r],label=f'{int(r*100)}%')
ax[0].set_xlabel('year'); ax[0].set_ylabel("PV of that year's $1,000")
ax[0].set_title('What one year is worth'); ax[0].legend(title='discount rate')
for r in rates:
    ax[1].plot(range(1,T100+1),cum[r],lw=2.8,color=cols[r],
               label=f'{int(r*100)}%  (cap {C/r:,.0f})')
    ax[1].axhline(C/r,ls=':',lw=1.4,color=cols[r])
    hit=next(t for t in range(1,T100+1) if cum[r][t-1]>=0.90*C/r)
    ax[1].plot([hit],[cum[r][hit-1]],'o',ms=11,color=cols[r],mec='w',mew=1.6)
    ax[1].annotate(f'90% by yr {hit}',xy=(hit,cum[r][hit-1]),
                   xytext=(hit+5,cum[r][hit-1]-2600),fontsize=10.5,color=cols[r],
                   arrowprops=dict(arrowstyle='->',color=cols[r]))
ax[1].set_xlabel('year'); ax[1].set_ylabel('cumulative NPV')
ax[1].set_title('Dotted = perpetuity cap'); ax[1].legend(fontsize=10,loc='lower right')
plt.tight_layout(); plt.show()


Read the "effective model" column — it is the punchline:

- **10% is a ~25-year model.** Years 50–100 are worth \$85 in total, less than one
  undiscounted year. Solving past ~40 years is wasted binaries.
- **5% is a ~48-year model.** Half the objective is decided in the first 14 years.
- **3% never really converges.** Only 74% of the perpetuity cap by year 45; years 50–100
  still carry \$5,869, about 19% of the total. Truncating a 3% model at 50 years *without* a
  terminal value term materially distorts it.

Two consequences that drive §4–§7:

1. **Late-horizon decisions are nearly free to the solver**, so the discount factor will not
   discipline them. If you care about end-of-horizon behaviour you need explicit terminal
   conditions — annualized capex (§4), a salvage term, or a cool-down buffer.
2. **The remaining tail past year $T$ is roughly $PV(T)/r$.** At 5% and $T=40$ that's
   $142/0.05 \approx \$2{,}840$, or 14% of the total. If two candidate plans differ by less
   than that, your truncation is driving the answer rather than the economics.

And note what this does *not* license: picking a discount rate to make a chosen horizon look
self-justifying. The rate comes from financing reality (real WACC, or a social discount rate
for policy work); the horizon comes from decision relevance. This table is a diagnostic for
*reporting*, not a design rule.


## 4. Lump-sum vs annualized capex

A 20-year asset built in year 16 of a 20-year model delivers 5 years of service inside the
horizon. Charge its **full** capital cost at the build year and the model sees a terrible
deal — so it refuses to build late and absorbs the shortfall instead.

Watch the `unmet_units` column.

In [ ]:
d = Data(T=20)
iy3 = list(range(1, 21, 3))
rows = []
for mode in ['lumpsum', 'annualized']:
    r = solve(d, mode, invest_years=iy3, capex_mode=mode)
    p = r['plan']
    rows.append(dict(mode=mode, objective=round(r['obj'], 1),
                     builds=sum(p.values()),
                     late_builds=sum(n for (s, v), n in p.items() if v >= 13),
                     unmet_units=round(r['slack'], 1),
                     build_years=sorted(set(v for s, v in p))))
pd.DataFrame(rows)

In [ ]:
# Why: value captured inside the horizon for a 20-yr asset, by build year
print("build_yr  service_yrs_in_horizon  value_captured")
for b in [1, 5, 9, 13, 16, 18]:
    last = min(b + d.life - 1, d.T)
    inh = sum(d.df[t] for t in range(b, last + 1) if t in d.df)
    full = sum(1/(1+d.r)**t for t in range(b, b + d.life))
    print(f"{b:8d}  {max(0,last-b+1):20d}  {100*inh/full:13.1f}%")

In [ ]:
# ---- shared plot style (run once) ----
import matplotlib.pyplot as plt, numpy as np
CB = {'blue':'#2E6F9E','orange':'#D97A2B','green':'#3F8F5B',
      'red':'#C0392B','grey':'#7F8C8D','purple':'#7D5BA6'}
plt.rcParams.update({'font.size':12,'axes.titlesize':13,'figure.dpi':120,
                     'axes.grid':True,'grid.alpha':0.3,
                     'axes.spines.top':False,'axes.spines.right':False})

In [ ]:
# FIG 1 -- value captured vs PV charged
fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
bys = np.arange(1, 19); vals = []
for b in bys:
    last = min(b + d.life - 1, d.T)
    inh = sum(d.df[t] for t in range(b, last+1) if t in d.df)
    full = sum(1/(1+d.r)**t for t in range(b, b+d.life))
    vals.append(100*inh/full)
ax[0].fill_between(bys, vals, color=CB['blue'], alpha=.18)
ax[0].plot(bys, vals, '-o', color=CB['blue'], lw=2.2, ms=5)
ax[0].annotate('late builds deliver\nalmost nothing', xy=(16, vals[15]), xytext=(9.5, 33),
               fontsize=10, color=CB['red'],
               arrowprops=dict(arrowstyle='->', color=CB['red'], lw=1.3))
ax[0].set(xlabel='build year', ylabel='% of full-life value',
          title=f'Value captured inside horizon\n({d.life}-yr asset, T={d.T}, r={d.r:.0%})')
for mode, c, mk, lab in [('lumpsum', CB['red'], 's', 'lump-sum'),
                         ('annualized', CB['green'], 'o', 'annualized (CRF)')]:
    ax[1].plot(bys, [capex_pv_multiplier(d, 'F1', b, mode) for b in bys],
               '-'+mk, color=c, lw=2.2, ms=5, label=lab)
ax[1].set(xlabel='build year', ylabel='PV multiplier', title='PV charged per $1 of capex')
ax[1].legend(fontsize=10)
plt.suptitle('Fig 1 -- End-of-horizon truncation bias', fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# FIG 2 -- where the builds actually land
order = ['M1','M2','P1','P2','F1','F2']
res = {m: solve(d, m, invest_years=iy3, capex_mode=m) for m in ['lumpsum','annualized']}
fig, axes = plt.subplots(1, 2, figsize=(11, 4.4), sharey=True)
for axi, (m, c) in zip(axes, [('lumpsum', CB['red']), ('annualized', CB['green'])]):
    for (s, v), n in res[m]['plan'].items():
        axi.scatter([v], [order.index(s)], s=260+120*n, color=c, zorder=3,
                    edgecolor='k', lw=.6)
        axi.text(v, order.index(s), str(n), ha='center', va='center', color='w',
                 fontweight='bold', fontsize=10, zorder=4)
    axi.set(xticks=iy3, yticks=range(6), xlabel='decision year',
            xlim=(0, 20.5), ylim=(-0.8, 5.8))
    axi.set_title(f"{m}\nobj={res[m]['obj']:,.0f}   unmet={res[m]['slack']:.0f} units")
axes[0].set_yticklabels(order)
axes[0].annotate('no builds in the\nfinal period', xy=(16, 4.9), xytext=(17.6, 2.6),
                 fontsize=10, color=CB['red'], ha='center',
                 arrowprops=dict(arrowstyle='->', color=CB['red'], lw=1.4))
plt.suptitle('Fig 2 -- Lump-sum capex refuses to build late (bubble = units)',
             fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

Lump-sum builds **nothing at all in the last investment period** and absorbs an order of
magnitude more unmet demand. That is not economics — it is an accounting artifact of
truncating the horizon while charging the full asset cost.

Two caveats for a real model: use the **same $r$** in the CRF as in the objective or you
reintroduce a wedge; and **lock the capital charge at the vintage's cost** — don't let it
float down as later learning occurs.

One honest counter-caveat: annualizing removes the anti-late-build bias so completely that
you'll see builds right at the horizon edge. That's correct if capacity can effectively be
rented, but if it's lumpy and irreversible you want an explicit salvage term or a
**cool-down buffer** — model to year 30, report to year 20.

### Visual: cost charged vs value delivered

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams.update({'font.size':12,'axes.grid':True,'grid.alpha':0.3})
yrs = list(range(1, d.T-1))
fig, ax = plt.subplots(1, 2, figsize=(12, 4.4))
ax[0].plot(yrs,[capex_pv_multiplier(d,'F1',v,'lumpsum') for v in yrs],'o-',lw=2.5,
           label='lump-sum',color='#c0392b')
ax[0].plot(yrs,[capex_pv_multiplier(d,'F1',v,'annualized') for v in yrs],'s-',lw=2.5,
           label='annualized (CRF)',color='#2471a3')
ax[0].set_xlabel('build decision year'); ax[0].set_ylabel('PV of $1 of capex')
ax[0].set_title('Cost charged inside the horizon'); ax[0].legend()
vc=[]
for b in yrs:
    on=b+d.lead['F1']; last=min(on+d.life-1,d.T)
    inh=sum(d.df[t] for t in range(on,last+1) if t in d.df)
    full=sum(1/(1+d.r)**t for t in range(on,on+d.life)); vc.append(100*inh/full)
ax[1].plot(yrs,vc,'D-',lw=2.5,color='#196f3d'); ax[1].fill_between(yrs,vc,alpha=.15,color='#196f3d')
ax[1].axhline(100,ls='--',lw=2,color='#c0392b')
ax[1].text(2.5,102,'lump-sum charges 100% regardless',fontsize=11,color='#c0392b')
ax[1].set_ylim(0,118); ax[1].set_xlabel('build decision year')
ax[1].set_ylabel('% of asset value captured'); ax[1].set_title('Value delivered inside the horizon')
plt.tight_layout(); plt.show()

## 5. Investment period granularity

Foresight and time granularity are **orthogonal axes**. A perfect-foresight MILP can have
5-year investment periods and annual operations; nothing about a monolithic solve forces the
indices to match.

Coarsening the investment index **restricts** the feasible set, so the objective is weakly
worse — every coarse result is a **valid upper bound** on the true optimum. (Coarsening
*operations* runs the other way: averaging flattens peaks, understating cost and
underbuilding capacity. Coarse investment + fine operations is the safe combination.)

In [ ]:
schemes = [('annual',    list(range(1, 21))),
           ('every 2yr', list(range(1, 21, 2))),
           ('every 3yr', list(range(1, 21, 3))),
           ('every 5yr', list(range(1, 21, 5))),
           ('staggered', staggered_years(20, fine=6, mid_step=2, mid_end=12, coarse_step=5))]
rows = []
for tag, iy in schemes:
    r = solve(d, tag, invest_years=iy)
    rows.append(dict(scheme=tag, n_periods=len(iy), binaries=r['nbin'],
                     objective=round(r['obj'], 1), nodes=r['nodes'],
                     seconds=round(r['sec'], 2)))
rowsB = rows
dfB = pd.DataFrame(rows)
dfB['excess_%'] = (100*(dfB.objective - dfB.objective.min())/dfB.objective.min()).round(2)
dfB

In [ ]:
# LP relaxation gives a lower bound; the coarse MILP an upper bound. Bracket the truth.
lp = build(d, invest_years=list(range(1, 21))); lp.update()
rel = lp.relax(); rel.Params.OutputFlag = 0; rel.optimize()
ub = dfB.loc[dfB.scheme == 'staggered', 'objective'].iloc[0]
print(f"LP lower bound       {rel.ObjVal:10.1f}")
print(f"staggered MILP (UB)  {ub:10.1f}")
print(f"bracket width        {100*(ub-rel.ObjVal)/ub:9.2f}%")

In [ ]:
# FIG 3 -- accuracy / speed frontier
G = dfB.to_dict('records'); base = min(g['objective'] for g in G)
offs = {'annual':(8,-4),'every 2yr':(6,10),'every 3yr':(8,0),
        'every 5yr':(8,0),'staggered':(6,-14)}
fig, ax = plt.subplots(1, 2, figsize=(11, 4.4))
for g in G:
    e = 100*(g['objective']-base)/base
    ax[0].scatter([g['seconds']], [e], s=170, color=CB['purple'], edgecolor='k', zorder=3)
    ax[0].annotate(g['scheme'], (g['seconds'], e), textcoords='offset points',
                   xytext=offs.get(g['scheme'], (8, 0)), fontsize=10)
ax[0].set_xscale('log')
ax[0].set(xlabel='solve time (s, log)', ylabel='excess cost vs best (%)',
          title='Accuracy / speed frontier')
tags = [g['scheme'] for g in G]
ax[1].bar(tags, [g['binaries'] for g in G],
          color=[CB['purple'] if t=='staggered' else CB['blue'] for t in tags])
for i, g in enumerate(G):
    ax[1].text(i, g['binaries']+8, f"{100*(g['objective']-base)/base:.1f}%",
               ha='center', fontsize=10)
ax[1].set(ylabel='binary variables', title='Binaries by scheme (label = excess cost)')
plt.setp(ax[1].get_xticklabels(), rotation=20, ha='right')
plt.suptitle('Fig 3 -- Coarsening is bounded, and cheap', fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

Note the shape: **staggered** (annual early, coarse late) buys most of the speedup of
uniform 5-year periods at a fraction of the accuracy cost, because it puts binaries where
decisions actually bind and where cost projections are credible.

This is the highest-leverage tractability move that **preserves perfect foresight** — try
it before reaching for rolling horizon.

### Visual: accuracy / size frontier

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.4))
cols=['#2471a3','#5499c7','#7fb3d5','#c0392b','#196f3d']
dfB = pd.DataFrame(rowsB)
bmin=dfB.objective.min()
for (_,row),c in zip(dfB.iterrows(),cols):
    ax[0].scatter(row.binaries,100*(row.objective-bmin)/bmin,s=220,color=c,zorder=3,
                  edgecolor='w',lw=1.5)
    ax[0].annotate(row.scheme,(row.binaries,100*(row.objective-bmin)/bmin),
                   textcoords='offset points',xytext=(0,14),ha='center',fontsize=11)
ax[0].set_xlabel('binary variables'); ax[0].set_ylabel('excess cost vs best (%)')
ax[0].set_title('Coarsening is always an upper bound'); ax[0].invert_xaxis()
ax[1].barh(dfB.scheme, dfB.seconds, color=cols); ax[1].set_xscale('log')
ax[1].set_xlabel('solve seconds (log)'); ax[1].set_title('Solve time')
plt.tight_layout(); plt.show()

## 6. Learning: none / exogenous / endogenous

- **exogenous** — capex declines with the *calendar*. The model gets cheaper technology
  whether or not it builds anything.
- **endogenous** — capex declines with *cumulative deployment*. The reduction must be
  bought down.

Watch which is cheaper.

In [ ]:
rows = []
for mode in ['none', 'exogenous', 'endogenous']:
    r = solve(d, mode, invest_years=iy3, learning=mode)
    p = r['plan']
    rows.append(dict(learning=mode, objective=round(r['obj'], 1),
                     builds=sum(p.values()),
                     early_builds=sum(n for (s, v), n in p.items() if v <= 7),
                     variables=r['nvar'], seconds=round(r['sec'], 2)))
pd.DataFrame(rows)

**Exogenous is the cheapest of the three — that is the free lunch.** It hands the model a
cost reduction requiring no deployment, which systematically biases toward *waiting*.
Endogenous sits between "none" and "exogenous": real, but earned.

In this instance build *timing* barely shifts, because legacy retirements dominate the
learning signal. Raise `d.LR` toward 0.35 and re-run to see timing move — that sensitivity
is itself the finding. **Your model is usually more sensitive to the learning rate than to
the discount rate**, and the learning rate has the weaker empirical grounding.

In [ ]:
print("Effective discount rate on CAPEX TIMING:  r_eff = (1+r)/(1-g) - 1\n")
print("   g \\ r     3%      5%      7%     10%")
for g in [0.00, 0.01, 0.02, 0.03, 0.05]:
    print(f"   {g*100:4.0f}%  " + "  ".join(
        f"{100*((1+r)/(1-g)-1):6.2f}" for r in [0.03, 0.05, 0.07, 0.10]))
print()
print("DIAGNOSTIC ONLY. It holds for the timing of one unit of capex, all else fixed.")
print("Swapping r=5%,g=3% for a flat r=8.25% would also over-discount opex, transport")
print("and demand-service value -- which learning never touched.")

In [ ]:
# FIG 4 -- why SOS2 is mandatory
Qbp, Cbp, unit = learning_breakpoints(d, 1600)
qs = np.linspace(d.Q0, 1600, 250)
fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
ax[0].plot(qs, [unit(q) for q in qs], lw=2.5, color=CB['blue'])
ax[0].axhline(d.c_floor_frac, ls='--', lw=2, color=CB['red'])
ax[0].text(430, d.c_floor_frac+.012, 'floor', color=CB['red'], fontsize=10)
ax[0].set(xlabel='cumulative capacity Q', ylabel='unit cost multiplier',
          title=f"Wright's law unit cost, LR={d.LR:.0%}\n(what you must NOT linearize)")
pw = [np.interp(q, Qbp, Cbp) for q in qs]
chord = [Cbp[0] + (Cbp[-1]-Cbp[0])*(q-Qbp[0])/(Qbp[-1]-Qbp[0]) for q in qs]
ax[1].fill_between(qs, chord, pw, color=CB['red'], alpha=.15,
                   label='unearned discount\nif SOS2 omitted')
ax[1].plot(qs, pw, lw=3, color=CB['blue'], label='true C(Q) -- concave')
ax[1].plot(Qbp, Cbp, '-o', lw=1.8, color=CB['green'], ms=7, label='SOS2 piecewise')
ax[1].plot(qs, chord, '--', lw=2.2, color=CB['red'], label='global chord (LP cheats here)')
ax[1].set(xlabel='cumulative capacity Q', ylabel='cumulative cost C(Q)',
          title='Linearize the CUMULATIVE cost')
ax[1].legend(fontsize=8.5, loc='upper left')
plt.suptitle('Fig 4 -- Why SOS2 is mandatory under minimization',
             fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()


### Checking the SOS2 mesh actually binds

A piecewise approximation can be silently useless: if $\lambda$ always lands on a single
breakpoint, you have a step function, not a curve. Verify it, and check where the
breakpoints sit relative to where the solution actually goes.


In [ ]:
m = build(d, invest_years=iy3, learning='endogenous'); m.optimize()
lam, Q = {}, {}
for v in m.getVars():
    if v.VarName.startswith('lam_'):
        _, yr, j = v.VarName.split('_'); lam.setdefault(int(yr), {})[int(j)] = v.X
    if v.VarName.startswith('Qcum_'):
        Q[int(v.VarName.split('_')[1])] = v.X
LS = set(d.learn_sites)
qmax = d.Q0 + sum(d.cap_unit[s]*d.max_builds for s in LS)*max(1, len(iy3)//3)
Qbp, _, _ = learning_breakpoints(d, qmax)
print('breakpoint grid:', [round(q) for q in Qbp])
print(f'solution reaches Q = {max(Q.values()):.0f}  of  qmax = {qmax:.0f}\n')
print('yr   Qcum   nonzero lambdas            status')
for yr in sorted(lam):
    nz = {j: round(x, 3) for j, x in lam[yr].items() if x > 1e-6}
    ks = sorted(nz)
    ok = 'interpolating' if len(ks) == 2 and ks[1]-ks[0] == 1 else (
         'at a breakpoint' if len(ks) == 1 else 'ADJACENCY VIOLATION')
    print(f'{yr:3d} {Q[yr]:7.1f}  {str(nz):26s} {ok}')
unused = sum(1 for j in range(len(Qbp)) if all(lam[yr].get(j, 0) < 1e-6 for yr in lam))
print(f'\nbreakpoints never used: {unused} of {len(Qbp)}')


The mesh binds correctly — several periods interpolate between **adjacent** pairs and
adjacency is never violated, which is exactly what SOS2 is there to enforce.

But note the last line: **several breakpoints are never used.** `qmax` is derived from a
worst-case buildout (every site, every unit, every period), which is far beyond where the
solution actually goes — so the mesh is coarsest precisely in the region that matters.
Tightening `qmax` to ~1.15× the realised $Q$ and re-solving raises the objective slightly,
because the coarse chords were understating the concave true cost. Small here (~0.03%), but
it scales with mesh coarseness and it always errs in the *optimistic* direction.

Practical rule: **solve once, read off the realised $Q$, then re-mesh around it.** One extra
solve for a mesh that's concentrated where the answer lives.


The three parameters cannot cancel because they live in different parts of the LP:

| Parameter | Lives in | Scope |
|---|---|---|
| Discount factor | Objective **weight** | Everything at time $t$ |
| Learning | Objective **cost vector** | New-build capex only |
| Efficiency | Constraint **matrix** | The feasible region |

**Double-count check:** if your capex trend was calibrated on \$/unit-*output*, efficiency
gains are already baked in and applying $\eta(v)$ on top counts them twice.
\$/unit-*capacity* is clean.

### Visual: vintage efficiency never crosses

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.6))
cm = plt.cm.viridis
for j,v in enumerate([-3,1,5,9,13,17]):
    ts=[t for t in range(max(1,v),d.T+1)]
    ax.plot(ts,[d.eta('P',v,t) for t in ts],'o-',lw=2.4,ms=5,color=cm(j/5),label=f'vintage {v}')
ax.axhline(d.eta_bar['P'],ls='--',lw=2,color='k')
ax.text(1.5,d.eta_bar['P']+0.004,r'ceiling $\bar\eta$',fontsize=12)
ax.set_xlabel('operating year'); ax.set_ylabel(r'processing yield $\eta(v,t)$')
ax.set_title('Older assets improve but never overtake newer vintages')
ax.legend(fontsize=10,ncol=2); plt.tight_layout(); plt.show()

## 7. Perfect foresight vs rolling horizon

**With identical deterministic data and a complete state handoff, a rolling sequence with
$W=T$ reproduces the PF solution exactly.** Bellman optimality: the continuation of solve
1's plan is feasible for solve 2 and must be optimal for it, or solve 1 wasn't optimal.
This holds despite MILP nonconvexity — deterministic DP decomposition needs a *sufficient
state*, not convexity.

Which means **full-window rolling buys nothing**: solve 1 is PF-sized, then you do it again.
All tractability gain comes from $W<T$.

The policy is evaluated by fixing the committed plan in the **full-horizon** model — you
evaluate the policy, you don't sum the myopic objectives.

In [ ]:
pf = solve(d, 'PF', invest_years=iy3)
print(f"PF objective                   {pf['obj']:10.1f}")
print(f"PF plan re-evaluated (sanity)  {evaluate_plan(d, pf['plan']):10.1f}\n")
rows = []
for W in [20, 12, 9, 6, 4]:
    plan, _ = rolling_horizon(d, W=W, delta=3, invest_step=3)
    val = evaluate_plan(d, plan)
    rows.append(dict(W=W, evaluated_cost=round(val, 1),
                     vs_PF_pct=round(100*(val-pf['obj'])/pf['obj'], 2),
                     builds=sum(plan.values())))
rowsW = rows
dfW = pd.DataFrame(rows)
dfW

$W=20$ reproduces PF. Mid-range $W$ differences land **inside the 0.5% MIP gap** — which is
the caution worth internalising: a single PF solve at a 1% gap gives a genuine 1% bound on
the whole answer, whereas a *sequence* of 1%-gap solves compounds error across rolls and
gives **no bound at all**.

Then $W=4$ falls off a cliff. That is the **hard floor**: $W$ must exceed lead time plus
enough operating years for the annuity to register, or long-lead assets never get built and
you misread an artifact as a result.

### The nested decision window — and the artifact to avoid

A decision zone shorter than the foresight window confines binaries to the near term while
the tail runs as pure LP. That is where rolling-horizon speedup actually lives.

But **do not hard-prohibit investment in the tail.** Forcing many years of capacity growth
into a few years of building causes systematic over-investment — and it's a *belief
inconsistency*: solve 1 assumes it can never build after the zone ends, while solves 2 and 3
demonstrably will.

In [ ]:
rows = []
for dz in [3, 6, 9]:
    for tail in [True, False]:
        plan, _ = rolling_horizon(d, W=12, delta=3, invest_step=3,
                                  decision_zone=dz, tail_continuous=tail)
        val = evaluate_plan(d, plan)
        rows.append(dict(decision_zone=dz, tail='continuous' if tail else 'BANNED',
                         evaluated_cost=round(val, 1),
                         vs_PF_pct=round(100*(val-pf['obj'])/pf['obj'], 2),
                         early_builds=sum(n for (s, v), n in plan.items() if v <= 7)))
rowsE = rows
dfE = pd.DataFrame(rows)
dfE

In [ ]:
# FIG 5 -- foresight floor + tail artifact
Wl = [20, 12, 9, 6, 4]
gaps = [dfW.loc[dfW.W == w, 'vs_PF_pct'].iloc[0] for w in Wl]
fig, ax = plt.subplots(1, 2, figsize=(11, 4.4))
ax[0].plot(range(5), gaps, '-o', lw=2.5, ms=9, color=CB['blue'])
ax[0].axhspan(-.5, .5, color=CB['grey'], alpha=.22); ax[0].axhline(0, color='k', lw=.8)
ax[0].text(.03, .42, 'inside 0.5% MIP gap', transform=ax[0].transAxes,
           fontsize=9.5, color='dimgrey')
ax[0].annotate('W < lead time\n+ annuity', xy=(4, gaps[-1]), xytext=(2.1, 8.5),
               fontsize=10.5, color=CB['red'],
               arrowprops=dict(arrowstyle='->', color=CB['red'], lw=1.4))
ax[0].set(xticks=range(5), xticklabels=Wl, ylim=(-2, 16),
          xlabel='foresight window W (decreasing)',
          ylabel='cost vs perfect foresight (%)')
ax[0].set_title('W=T reproduces PF;\nbelow the floor it collapses')
x = np.arange(3); w = .36
sel = lambda z, tl, col: dfE[(dfE.decision_zone == z) & (dfE['tail'] == tl)][col].iloc[0]
cv = [sel(z, 'continuous', 'vs_PF_pct') for z in [3,6,9]]
bv = [sel(z, 'BANNED', 'vs_PF_pct') for z in [3,6,9]]
ce = [sel(z, 'continuous', 'early_builds') for z in [3,6,9]]
be = [sel(z, 'BANNED', 'early_builds') for z in [3,6,9]]
ax[1].bar(x-w/2, cv, w, color=CB['green'], label='tail continuous')
ax[1].bar(x+w/2, bv, w, color=CB['red'], label='tail BANNED')
for i in range(3):
    ax[1].text(i-w/2, cv[i]+.6, f'{ce[i]} early', ha='center', fontsize=9)
    ax[1].text(i+w/2, bv[i]+.6, f'{be[i]} early', ha='center', fontsize=9, fontweight='bold')
ax[1].set(xticks=x, xticklabels=[f'zone={z}yr' for z in [3,6,9]], ylim=(-2, 27),
          ylabel='cost vs PF (%)')
ax[1].set_title('Banning tail investment\n= over-investment artifact')
ax[1].legend(fontsize=10, loc='upper right')
plt.suptitle('Fig 5 -- Rolling horizon: foresight floor and the tail artifact',
             fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# FIG 6 -- what the model actually does
mm = build(d, invest_years=iy3); mm.optimize()
tiers = {'Mining': d.mines, 'Processing': d.procs, 'Manufacturing': d.fabs}
cap = {k: [] for k in tiers}
for t in d.years:
    for k, ss in tiers.items():
        tot = 0.
        for s in ss:
            if s in d.legacy:
                ln, lv, lret = d.legacy[s]
                if t <= lret: tot += ln*d.cap_unit[s]
            for (sv, v, kk), var in mm._y.items():
                if sv == s and var.X > 0.5:
                    on = v + d.lead[s]
                    if on <= t <= on + d.life - 1: tot += d.cap_unit[s]
        cap[k].append(tot)
fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
for k, c in zip(tiers, [CB['orange'], CB['blue'], CB['purple']]):
    ax[0].plot(d.years, cap[k], '-o', lw=2.5, ms=4, color=c, label=k)
for s, (n, v, ret) in d.legacy.items():
    ax[0].axvline(ret, ls=':', c='grey', lw=.8)
ax[0].set(xlabel='year', ylabel='installed capacity (units)')
ax[0].set_title('Replacement waves as legacy retires\n(dotted = retirement years)')
ax[0].legend(fontsize=10)
d1 = [d.D['R1', t] for t in d.years]; d2 = [d.D['R2', t] for t in d.years]
served = [sum(mm._ffr[f, g, t].X for f in d.fabs for g in d.regions) for t in d.years]
ax[1].stackplot(d.years, d1, d2, colors=['#9ecae1', '#f4a582'],
                labels=['Region 1 demand', 'Region 2 demand'], alpha=.9)
ax[1].plot(d.years, served, 'k-', lw=2.5, label='delivered')
ax[1].set(xlabel='year', ylabel='units', title='Region 2 growth drives the build-out')
ax[1].legend(fontsize=9, loc='upper left')
plt.suptitle('What the model actually does', fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

With a 3-year decision zone, banning tail investment costs **~+22%** and roughly doubles
early builds. Relaxing the tail to *continuous* capacity — no binaries, no siting
lumpiness, but the model knows it can build later — eliminates the artifact while keeping
the entire binary reduction.

Longer zones mask the problem entirely, which is exactly why it ships undetected.

### Visual: the foresight cliff and the tail-ban artifact

In [ ]:
import numpy as np
dfD = pd.DataFrame(rowsW)
dfE = pd.DataFrame(rowsE)
fig, ax = plt.subplots(1, 2, figsize=(12, 4.4))
ax[0].plot(dfD.W, dfD.vs_PF_pct,'o-',lw=3,ms=11,color='#2471a3')
ax[0].axhline(0,ls='--',color='#196f3d',lw=2)
ax[0].axhspan(-0.5,0.5,color='#196f3d',alpha=.10)
ax[0].text(13,-1.9,'inside the 0.5% MIP gap',fontsize=10.5,color='#196f3d')
ax[0].set_xlabel('foresight window W'); ax[0].set_ylabel('cost vs PF (%)')
ax[0].set_title('W=T reproduces PF; W=4 falls off a cliff'); ax[0].invert_xaxis()
if True:
    p = dfE.pivot(index='decision_zone',columns='tail',values='vs_PF_pct')
    x=np.arange(len(p)); w=0.36
    ax[1].bar(x-w/2,p['continuous'],w,label='tail continuous',color='#196f3d')
    ax[1].bar(x+w/2,p['BANNED'],w,label='tail BANNED',color='#c0392b')
    ax[1].set_xticks(x); ax[1].set_xticklabels([f'dz={i}' for i in p.index])
    ax[1].set_ylabel('cost vs PF (%)'); ax[1].legend()
    ax[1].set_title('Banning tail investment = over-build')
plt.tight_layout(); plt.show()

## 8. Summary — deterministic

| Choice | Direction of error | Fix |
|---|---|---|
| Lump-sum capex | Refuses late builds, absorbs shortfall | Annualize (CRF at the objective's $r$) |
| Coarse investment periods | Overstates cost — **valid upper bound** | Staggered; bracket with LP relaxation |
| Coarse operations | *Understates* cost, underbuilds | Keep operations fine |
| Exogenous learning | Free lunch; biases toward waiting | Endogenous, SOS2 on **cumulative** cost |
| $\eta$ indexed on $t$ only | Silently upgrades existing assets | Index on vintage $(v,t)$ |
| Rolling with $W=T$ | No speedup; MIP gaps compound | Use $W<T$, or stay with PF |
| $W$ < lead time + annuity | Long-lead assets never built | Hard floor on $W$ |
| Banned tail investment | Over-investment | Continuous tail capacity |

**Recommended order:** staggered investment periods + fine operations + relaxed integrality
in far periods + 1–2% MIP gap, all inside perfect foresight. Move to rolling horizon only
if that's insufficient — and if the real issue is **uncertainty** rather than size, go to
Part 2 instead.

### Things to try
- `d.LR = 0.35` in §5 — does build *timing* shift, not just cost?
- `d.lead = {s: 6 for s in d.sites}` in §6 — the $W$ cliff should move right
- `d.r = 0.10` throughout — late-horizon decisions become nearly weightless